### Proyecto Despliegue Soluciones Analiticas

Presentado por:
- Laura Marcela Ramirez Casallas
- Kevin Alejandro Caicedo Rodriguez
- Jhon Jairo Meneses Prieto
- Jeferson Andrés Gutiérrez Gutiérrez

## Pregunta de Negocio


El examen **saber 11°** es una prueba utilizada por el instituto colombiano para la evaluación de la Educación (ICFES)1, para realizar mediciones oficiales acerca de la calidad del servicio educativo prestado a nivel nacional. Esta prueba es practicada a los estudiantes de todos los establecimientos educativos, privados y públicos, que están terminando su formación académica media (grado 11, previo a iniciar la formación universitaria). 

Con este contexto, el problema que vamos a abordar se enfoca en entender cuáles son los factores o variables que inciden en los resultados obtenidos por cada estudiante con el fin de enfocar el uso de los recursos disponibles para implementar políticas públicas que busquen fortalecer las competencias de los estudiantes del país. 

Por tal razón, bajo este contexto, la pregunta de negocio es la siguiente:

#### *¿Cómo puede la secretaria de educacion lograr el aumento de la aprobación del ICFES mediante el análisis de variables demográficas, socioeconómicos e institucionales para identificar brechas de rendimiento y mejorar la calidad educativa de cada una de las instituciones analizadas de la ciudad de Bogotá?*

## Fuente de los datos

Para dar solución al caso de estudio contamos con los datos publicados a través de la plataforma de Datos Abiertos de Colombia, a los que se puede acceder a través del siguiente URL: 

https://www.datos.gov.co/en/Educaci-n/Bogota_historico_saber_11/8qwp-pzhz/about_data

## Importación de Datos

In [1]:
pip install Sodapy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


Vamos a utilizar el año 2022, que es el mas reciente y el que surge luego de pandemia

In [2]:
from sodapy import Socrata
import pandas as pd

APP_TOKEN = "zkj4BAQIZkr7n4t4NKzvLt5zq"
client = Socrata("datos.gov.co", APP_TOKEN, timeout=120)

dataset_id = "8qwp-pzhz"
periodo_objetivo = 20224

limit = 50000
offset = 0
all_results = []

while True:
    results = client.get(
        dataset_id,
        where=f"periodo = '{periodo_objetivo}'",
        order="estu_consecutivo ASC",
        limit=limit,
        offset=offset
    )

    if not results:
        break

    all_results.extend(results)
    offset += len(results)

    print(f"Descargados {len(all_results)} registros...")

df = pd.DataFrame.from_records(all_results)

print("Total de registros:", len(df))
print("Filas completamente duplicadas:", df.duplicated().sum())
print("Identificadores únicos:", df["estu_consecutivo"].nunique())

Descargados 50000 registros...
Descargados 100000 registros...
Descargados 150000 registros...
Descargados 152584 registros...
Total de registros: 152584
Filas completamente duplicadas: 76331
Identificadores únicos: 76253


In [3]:
print(f"Registros: {df.shape[0]:,}")
print(f"Variables: {df.shape[1]}")
display(df.head())

Registros: 152,584
Variables: 51


,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,cole_cod_depto_ubicacion,...,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,desemp_ingles,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
0,20224,TI,SB11202240002331,URBANO,N,A,ACADÉMICO,111001014176,111001014176,11,...,Si,Si,Si,A1,56,51,52,46,59,262
1,20224,TI,SB11202240002331,URBANO,N,A,ACADÉMICO,111001014176,111001014176,11,...,Si,Si,Si,A1,56,51,52,46,59,262
2,20224,TI,SB11202240002335,URBANO,N,A,ACADÉMICO,111001014176,111001014176,11,...,No,No,Si,A1,48,51,49,46,45,239
3,20224,TI,SB11202240002335,URBANO,N,A,ACADÉMICO,111001014176,111001014176,11,...,No,No,Si,A1,48,51,49,46,45,239
4,20224,TI,SB11202240002340,URBANO,N,A,ACADÉMICO,111001076376,111001076376,11,...,No,No,Si,A-,42,45,34,39,55,216


In [4]:
display(df.info())
display(df.describe(include="all").T)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 152584 entries, 0 to 152583
Data columns (total 51 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0   periodo                        152584 non-null  object
 1   estu_tipodocumento             152584 non-null  object
 2   estu_consecutivo               152584 non-null  object
 3   cole_area_ubicacion            152584 non-null  object
 4   cole_bilingue                  124416 non-null  object
 5   cole_calendario                152584 non-null  object
 6   cole_caracter                  145548 non-null  object
 7   cole_cod_dane_establecimiento  152584 non-null  object
 8   cole_cod_dane_sede             152584 non-null  object
 9   cole_cod_depto_ubicacion       152584 non-null  object
 10  cole_cod_mcpio_ubicacion       152584 non-null  object
 11  cole_codigo_icfes              152584 non-null  object
 12  cole_depto_ubicacion           152584 non-nu

None

,count,unique,top,freq
periodo,152584,1,20224,152584
estu_tipodocumento,152584,11,TI,131232
estu_consecutivo,152584,76253,SB11202240071815,4
cole_area_ubicacion,152584,2,URBANO,151258
cole_bilingue,124416,2,N,123762
cole_calendario,152584,3,A,151360
cole_caracter,145548,4,ACADÉMICO,124824
cole_cod_dane_establecimiento,152584,1131,111001024732,852
cole_cod_dane_sede,152584,1143,111001024732,852
cole_cod_depto_ubicacion,152584,1,11,152584


# Diagnósitico de calidad de datos

### Nulos x columnas

In [5]:
resumen_nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2)
})

resumen_nulos = resumen_nulos.query("nulos > 0")
resumen_nulos = resumen_nulos.sort_values(
    "porcentaje_nulos",
    ascending=False
)

display(resumen_nulos)

,nulos,porcentaje_nulos
cole_bilingue,28168,18.46
cole_caracter,7036,4.61
fami_estratovivienda,6514,4.27
fami_tieneinternet,6200,4.06
fami_educacionpadre,6108,4.00
fami_educacionmadre,6088,3.99
fami_tieneautomovil,4346,2.85
fami_cuartoshogar,4330,2.84
fami_tienelavadora,4182,2.74
fami_tienecomputador,4130,2.71


### Duplicidad

In [6]:
print('Filas duplicadas: ', df.duplicated().sum())
print("Duplicados por estudiante:",df["estu_consecutivo"].duplicated().sum())

Filas duplicadas:  76331
Duplicados por estudiante: 76331


In [7]:
# ver estudiantes que mas se repiten
df["estu_consecutivo"].value_counts().loc[lambda x: x > 1]

estu_consecutivo
SB11202240071815    4
SB11202240072159    4
SB11202240072128    4
SB11202240072129    4
SB11202240072133    4
                   ..
SB11202240169128    2
SB11202240169119    2
SB11202240169113    2
SB11202240169112    2
SB11202240616361    2
Name: count, Length: 76253, dtype: int64

In [8]:
pd.set_option('display.max_columns', None)

df_duplicados = df[
    df.duplicated(keep=False)
]

display(df_duplicados[df_duplicados['estu_consecutivo']=='SB11202240072129'])

,periodo,estu_tipodocumento,estu_consecutivo,cole_area_ubicacion,cole_bilingue,cole_calendario,cole_caracter,cole_cod_dane_establecimiento,cole_cod_dane_sede,cole_cod_depto_ubicacion,cole_cod_mcpio_ubicacion,cole_codigo_icfes,cole_depto_ubicacion,cole_genero,cole_jornada,cole_mcpio_ubicacion,cole_naturaleza,cole_nombre_establecimiento,cole_nombre_sede,cole_sede_principal,estu_cod_depto_presentacion,estu_cod_mcpio_presentacion,estu_cod_reside_depto,estu_cod_reside_mcpio,estu_depto_presentacion,estu_depto_reside,estu_estadoinvestigacion,estu_estudiante,estu_fechanacimiento,estu_genero,estu_mcpio_presentacion,estu_mcpio_reside,estu_nacionalidad,estu_pais_reside,estu_privado_libertad,fami_cuartoshogar,fami_educacionmadre,fami_educacionpadre,fami_estratovivienda,fami_personashogar,fami_tieneautomovil,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,desemp_ingles,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
21912,20224,TI,SB11202240072129,URBANO,N,A,ACADÉMICO,111001000612,111001000612,11,11001,023382,BOGOTÁ,FEMENINO,TARDE,BOGOTÁ D.C.,OFICIAL,COLEGIO FEMENINO LORENCITA VILLEGAS DE SANTOS ...,SEDE A,S,11,11001,11,11001,BOGOTÁ,BOGOTÁ,PUBLICAR,ESTUDIANTE,23/08/2004,F,BOGOTÁ D.C.,BOGOTÁ D.C.,COLOMBIA,COLOMBIA,N,Dos,Educación profesional completa,Secundaria (Bachillerato) incompleta,Estrato 3,1 a 2,Si,Si,Si,Si,A2,64,55,64,66,69,318
21913,20224,TI,SB11202240072129,URBANO,N,A,ACADÉMICO,111001000612,111001000612,11,11001,023382,BOGOTÁ,FEMENINO,TARDE,BOGOTÁ D.C.,OFICIAL,COLEGIO FEMENINO LORENCITA VILLEGAS DE SANTOS ...,SEDE A,S,11,11001,11,11001,BOGOTÁ,BOGOTÁ,PUBLICAR,ESTUDIANTE,23/08/2004,F,BOGOTÁ D.C.,BOGOTÁ D.C.,COLOMBIA,COLOMBIA,N,Dos,Educación profesional completa,Secundaria (Bachillerato) incompleta,Estrato 3,1 a 2,Si,Si,Si,Si,A2,64,55,64,66,69,318
21914,20224,TI,SB11202240072129,URBANO,N,A,ACADÉMICO,111001000612,111001000612,11,11001,023382,BOGOTÁ,FEMENINO,TARDE,BOGOTÁ D.C.,OFICIAL,COLEGIO FEMENINO LORENCITA VILLEGAS DE SANTOS ...,SEDE A,S,11,11001,11,11001,BOGOTÁ,BOGOTÁ,PUBLICAR,ESTUDIANTE,23/08/2004,F,BOGOTÁ D.C.,BOGOTÁ D.C.,COLOMBIA,COLOMBIA,N,Dos,Educación profesional completa,Secundaria (Bachillerato) incompleta,Estrato 3,1 a 2,Si,Si,Si,Si,A2,64,55,64,66,69,318
21915,20224,TI,SB11202240072129,URBANO,N,A,ACADÉMICO,111001000612,111001000612,11,11001,023382,BOGOTÁ,FEMENINO,TARDE,BOGOTÁ D.C.,OFICIAL,COLEGIO FEMENINO LORENCITA VILLEGAS DE SANTOS ...,SEDE A,S,11,11001,11,11001,BOGOTÁ,BOGOTÁ,PUBLICAR,ESTUDIANTE,23/08/2004,F,BOGOTÁ D.C.,BOGOTÁ D.C.,COLOMBIA,COLOMBIA,N,Dos,Educación profesional completa,Secundaria (Bachillerato) incompleta,Estrato 3,1 a 2,Si,Si,Si,Si,A2,64,55,64,66,69,318


Se confirma que la duplicidad es en todos los campos 

### Tipos de Datos

In [9]:
for columna in df.columns:
    print(f"{columna}: {df[columna].dtype}")

periodo: object
estu_tipodocumento: object
estu_consecutivo: object
cole_area_ubicacion: object
cole_bilingue: object
cole_calendario: object
cole_caracter: object
cole_cod_dane_establecimiento: object
cole_cod_dane_sede: object
cole_cod_depto_ubicacion: object
cole_cod_mcpio_ubicacion: object
cole_codigo_icfes: object
cole_depto_ubicacion: object
cole_genero: object
cole_jornada: object
cole_mcpio_ubicacion: object
cole_naturaleza: object
cole_nombre_establecimiento: object
cole_nombre_sede: object
cole_sede_principal: object
estu_cod_depto_presentacion: object
estu_cod_mcpio_presentacion: object
estu_cod_reside_depto: object
estu_cod_reside_mcpio: object
estu_depto_presentacion: object
estu_depto_reside: object
estu_estadoinvestigacion: object
estu_estudiante: object
estu_fechanacimiento: object
estu_genero: object
estu_mcpio_presentacion: object
estu_mcpio_reside: object
estu_nacionalidad: object
estu_pais_reside: object
estu_privado_libertad: object
fami_cuartoshogar: object
fami_e

Todas las columnas aparecen como object. Esto puede ocurrir porque los datos se descargaron desde Socrata como texto.



### Categorias unicas x cvariable

In [10]:
for col in df.columns:
    print(f'Columna: {col} , Valores Unicos: {df[col].unique()}')

Columna: periodo , Valores Unicos: ['20224']
Columna: estu_tipodocumento , Valores Unicos: ['TI' 'CC' 'CR' 'CE' 'PEP' 'NES' 'PPT' 'NUIP' 'RC' 'PE' 'PC']
Columna: estu_consecutivo , Valores Unicos: ['SB11202240002331' 'SB11202240002335' 'SB11202240002340' ...
 'SB11202240616309' 'SB11202240616360' 'SB11202240616361']
Columna: cole_area_ubicacion , Valores Unicos: ['URBANO' 'RURAL']
Columna: cole_bilingue , Valores Unicos: ['N' nan 'S']
Columna: cole_calendario , Valores Unicos: ['A' 'OTRO' 'B']
Columna: cole_caracter , Valores Unicos: ['ACADÉMICO' 'TÉCNICO/ACADÉMICO' nan 'TÉCNICO' 'NO APLICA']
Columna: cole_cod_dane_establecimiento , Valores Unicos: ['111001014176' '111001076376' '111001035572' ... '311001043159'
 '311001096783' '311001003271']
Columna: cole_cod_dane_sede , Valores Unicos: ['111001014176' '111001076376' '111001035572' ... '311001043159'
 '311001096783' '311001003271']
Columna: cole_cod_depto_ubicacion , Valores Unicos: ['11']
Columna: cole_cod_mcpio_ubicacion , Valores 

Con esta validacion podemos ir determinando que variables se repiten en informacion o tienen pocos valores, lo que puede hacerla candidata de eliminacion posterior

### Outliers  o valores inesperados

In [11]:
for columna in df.columns:
    if df[columna].dtype == "object":
        print(f"\n{columna}:")
        print(df[columna].value_counts(dropna=False).head(20))


periodo:
periodo
20224    152584
Name: count, dtype: int64

estu_tipodocumento:
estu_tipodocumento
TI      131232
CC       18578
NES        814
CE         718
PEP        564
CR         542
PPT         68
PE          54
NUIP         6
RC           4
PC           4
Name: count, dtype: int64

estu_consecutivo:
estu_consecutivo
SB11202240071815    4
SB11202240072159    4
SB11202240072128    4
SB11202240072129    4
SB11202240072133    4
SB11202240072137    4
SB11202240072140    4
SB11202240072145    4
SB11202240072155    4
SB11202240072157    4
SB11202240071460    4
SB11202240072161    4
SB11202240072116    4
SB11202240072164    4
SB11202240072165    4
SB11202240072166    4
SB11202240072168    4
SB11202240072170    4
SB11202240071420    4
SB11202240072171    4
Name: count, dtype: int64

cole_area_ubicacion:
cole_area_ubicacion
URBANO    151258
RURAL       1326
Name: count, dtype: int64

cole_bilingue:
cole_bilingue
N      123762
NaN     28168
S         654
Name: count, dtype: int64

cole_c

In [12]:
#validacion de puntajes
columnas_puntajes = [
    "punt_global",
    "punt_matematicas",
    "punt_sociales_ciudadanas",
    "punt_c_naturales",
    "punt_lectura_critica",
    "punt_ingles"
]

for columna in columnas_puntajes:
    puntajes = pd.to_numeric(df[columna], errors="coerce")

    print(f"\n{columna}")
    print("Mínimo:", puntajes.min())
    print("Máximo:", puntajes.max())
    print("Valores negativos:", (puntajes < 0).sum())
    print("Valores no numéricos:", (puntajes.isna() & df[columna].notna()).sum())


punt_global
Mínimo: 0
Máximo: 450
Valores negativos: 0
Valores no numéricos: 0

punt_matematicas
Mínimo: 0
Máximo: 100
Valores negativos: 0
Valores no numéricos: 0

punt_sociales_ciudadanas
Mínimo: 0
Máximo: 100
Valores negativos: 0
Valores no numéricos: 0

punt_c_naturales
Mínimo: 0
Máximo: 100
Valores negativos: 0
Valores no numéricos: 0

punt_lectura_critica
Mínimo: 0
Máximo: 100
Valores negativos: 0
Valores no numéricos: 0

punt_ingles
Mínimo: 0.0
Máximo: 100.0
Valores negativos: 0
Valores no numéricos: 0


Se confirma que variables como fecha de nacimiento es correcta y que los rangos en puntajes es el esperado

## Descarga Base Inicial

Se realiza descarga para guardar la primera version de los datos en DVC

In [13]:
df.to_csv("Saber11_2022.csv", index=False, encoding = "utf-8-sig")

In [14]:
print("Base original guardada.")
print(f"Registros originales: {len(df):,}")
print(f"Columnas originales: {df.shape[1]}")

Base original guardada.
Registros originales: 152,584
Columnas originales: 51


# Limpieza de Datos

In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
sns.set_theme(style="whitegrid")

## Eliminación de duplicados

In [16]:
print("Total de registros:", len(df))

print(
    "Estudiantes únicos:",
    df["estu_consecutivo"].nunique()
)

print(
    "Identificadores nulos:",
    df["estu_consecutivo"].isna().sum()
)

print(
    "Filas completamente duplicadas:",
    df.duplicated().sum()
)

Total de registros: 152584
Estudiantes únicos: 76253
Identificadores nulos: 0
Filas completamente duplicadas: 76331


En primer lugar y dado que se comprueba que los registro con el código de estudiante vienen desde la importación de los datos procedemos a eliminar los registros duplicados:

In [17]:
if "punt_global" not in df.columns:
    raise KeyError("No existe la columna 'punt_global' en la base.")

dffinal = df.loc[
    df["punt_global"].notna()
].copy()

print("\nELIMINACIÓN DE REGISTROS SIN PUNTAJE GLOBAL")
print(f"Registros iniciales: {len(dffinal):,}")
print(f"Registros sin puntaje global: {dffinal['punt_global'].isna().sum():,}")
print(f"Registros conservados: {len(dffinal):,}")


ELIMINACIÓN DE REGISTROS SIN PUNTAJE GLOBAL
Registros iniciales: 152,584
Registros sin puntaje global: 0
Registros conservados: 152,584


In [18]:
duplicados_eliminados = dffinal.duplicated().sum()

dffinal = dffinal.drop_duplicates().copy()

print("\nELIMINACIÓN DE DUPLICADOS EXACTOS")
print(f"Duplicados exactos eliminados: {duplicados_eliminados:,}")
print(f"Registros después de eliminar duplicados: {len(dffinal):,}")



ELIMINACIÓN DE DUPLICADOS EXACTOS
Duplicados exactos eliminados: 76,331
Registros después de eliminar duplicados: 76,253


In [19]:
dffinal.info()

<class 'pandas.core.frame.DataFrame'>
Index: 76253 entries, 0 to 152582
Data columns (total 51 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   periodo                        76253 non-null  object
 1   estu_tipodocumento             76253 non-null  object
 2   estu_consecutivo               76253 non-null  object
 3   cole_area_ubicacion            76253 non-null  object
 4   cole_bilingue                  62169 non-null  object
 5   cole_calendario                76253 non-null  object
 6   cole_caracter                  72735 non-null  object
 7   cole_cod_dane_establecimiento  76253 non-null  object
 8   cole_cod_dane_sede             76253 non-null  object
 9   cole_cod_depto_ubicacion       76253 non-null  object
 10  cole_cod_mcpio_ubicacion       76253 non-null  object
 11  cole_codigo_icfes              76253 non-null  object
 12  cole_depto_ubicacion           76253 non-null  object
 13  cole_

## Eliminación de columnas no requeridas para el modelo

Descartaremos las columnas relacionadas con códigos de identificación (DANE // colegio) dado que podemos asociarlas por el nombre del código que representan:
- estu_tipodocumento
- estu_consecutivo
- estu_estadoinvestigacion
- cole_cod_dane_establecimiento  
- cole_cod_dane_sede  
- cole_cod_depto_ubicacion  
- cole_cod_mcpio_ubicacion  
- estu_cod_reside_depto  
- estu_cod_reside_mcpio
- estu_cod_reside_depto
- estu_cod_reside_mcpio
- cole_codigo_icfes - se descarta puesto que es un identificador de cada colegio

Adicionalmente eliminaremos las variables relacionadas con información sobre los colegios teniendo en consideración que no brinda información adicional con relación a la identificación del colegio principal:

- cole_nombre_establecimiento  
- cole_nombre_sede  
- cole_sede_principal

Eliminaremos la variable **Estu_estudiante** dado que solo contiene un valor unico, la variable cole_cod_mcpio_ubicacion dado que todos los estudiantes son de Bogotá.

Prescindiremos de la columna estu_fechanacimiento dado que el rango de edad de los estudiantes por la naturaleza del evento no deberia presentar variaciones significativas.

Sumado a esto, descartaremos las variables: estu_depto_reside, estu_pais_reside, estu_depto_presentacion, estu_mcpio_presentacion, estu_mcpio_reside, estu_nacionalidad teniendo en cuenta que el enfoque de estudio está orientado a estudiantes de colegios ubicados en la ciudad de Bogotá.

In [21]:
# Quitar columnas:
columnas_a_quitar = ['estu_tipodocumento','estu_fechanacimiento','estu_consecutivo','cole_mcpio_ubicacion','estu_estadoinvestigacion','cole_cod_dane_establecimiento', 'cole_cod_dane_sede','cole_cod_depto_ubicacion','cole_cod_mcpio_ubicacion','estu_cod_reside_depto','estu_cod_reside_mcpio','estu_cod_reside_depto','estu_cod_depto_presentacion','estu_cod_mcpio_presentacion','estu_cod_reside_mcpio','cole_codigo_icfes','cole_nombre_establecimiento','cole_nombre_sede','cole_sede_principal','estu_estudiante','cole_depto_ubicacion','estu_depto_presentacion','cole_cod_mcpio_ubicacion','estu_pais_reside','estu_mcpio_reside','estu_mcpio_presentacion','estu_depto_reside','estu_nacionalidad','estu_privado_libertad']
dffinal.drop(columns=columnas_a_quitar, inplace=True)

In [22]:
dffinal.shape

(76253, 25)

In [23]:
for columna in dffinal.columns:
    print(columna)

periodo
cole_area_ubicacion
cole_bilingue
cole_calendario
cole_caracter
cole_genero
cole_jornada
cole_naturaleza
estu_genero
fami_cuartoshogar
fami_educacionmadre
fami_educacionpadre
fami_estratovivienda
fami_personashogar
fami_tieneautomovil
fami_tienecomputador
fami_tieneinternet
fami_tienelavadora
desemp_ingles
punt_ingles
punt_matematicas
punt_sociales_ciudadanas
punt_c_naturales
punt_lectura_critica
punt_global


Pasamos de tener 51 variables a 25 con información relevante para el modelo

## Tratamiento de nulos

In [24]:
resumen_nulos = pd.DataFrame({
    "nulos": dffinal.isna().sum(),
    "porcentaje_nulos": (dffinal.isna().mean() * 100).round(2)
})

resumen_nulos = resumen_nulos.query("nulos > 0")
resumen_nulos = resumen_nulos.sort_values(
    "porcentaje_nulos",
    ascending=False
)

display(resumen_nulos)

,nulos,porcentaje_nulos
cole_bilingue,14084,18.47
cole_caracter,3518,4.61
fami_estratovivienda,3254,4.27
fami_tieneinternet,3097,4.06
fami_educacionpadre,3051,4.00
fami_educacionmadre,3041,3.99
fami_tieneautomovil,2173,2.85
fami_cuartoshogar,2165,2.84
fami_tienelavadora,2091,2.74
fami_tienecomputador,2065,2.71


In [25]:
# Variables categóricas del colegio: imputar moda si existe.
columnas_moda = ["cole_caracter","cole_bilingue","fami_estratovivienda","desemp_ingles"]

for columna in columnas_moda:
    if columna in dffinal.columns and dffinal[columna].notna().any():
        moda = dffinal[columna].mode(dropna=True)

        if not moda.empty:
            dffinal[columna] = dffinal[columna].fillna(moda.iloc[0])


# Género: identificar los faltantes sin asignarles una categoría real.
if "estu_genero" in dffinal.columns:
    dffinal["estu_genero"] = (dffinal["estu_genero"].fillna("N.D."))


# Variables de posesiones del hogar.
# No se asume que la ausencia de respuesta signifique "No".
columnas_posesiones = ["fami_tieneautomovil","fami_tienecomputador","fami_tieneinternet","fami_tienelavadora"]

for columna in columnas_posesiones:
    if columna in dffinal.columns:
        dffinal[columna] = (dffinal[columna].fillna("No informa"))


# Educación de madre y padre.
for columna in ["fami_educacionmadre", "fami_educacionpadre"]:
    if columna in dffinal.columns:
        dffinal[columna] = (dffinal[columna].fillna("No sabe"))


# Variables familiares con categorías de cantidad.
for columna in ["fami_cuartoshogar", "fami_personashogar"]:
    if columna in dffinal.columns:
        dffinal[columna] = (dffinal[columna].fillna("No informa"))


In [26]:
if "fami_cuartoshogar" in dffinal.columns:
    reemplazos_cuartos = {"Seis": "Seis o más","Siete": "Seis o más","Ocho": "Seis o más","Nueve": "Seis o más","Diez o más": "Seis o más"}
    dffinal["fami_cuartoshogar"] = (dffinal["fami_cuartoshogar"].replace(reemplazos_cuartos))


# Agrupar tamaños del hogar.
if "fami_personashogar" in dffinal.columns:
    reemplazos_personas = {"Una": "1 a 2","Dos": "1 a 2","Tres": "3 a 4","Cuatro": "3 a 4","Cinco": "5 a 6","Seis": "5 a 6","Siete": "7 a 8","Ocho": "7 a 8","Nueve": "9 o más","Diez": "9 o más","Once": "9 o más","Doce o más": "9 o más"}

    dffinal["fami_personashogar"] = (dffinal["fami_personashogar"].replace(reemplazos_personas))

In [27]:
resumen_nulos = pd.DataFrame({
    "nulos": dffinal.isna().sum(),
    "porcentaje_nulos": (dffinal.isna().mean() * 100).round(2)
})

resumen_nulos = resumen_nulos.query("nulos > 0")
resumen_nulos = resumen_nulos.sort_values(
    "porcentaje_nulos",
    ascending=False
)

display(resumen_nulos)

,nulos,porcentaje_nulos
punt_ingles,15,0.02


In [28]:
display(dffinal["punt_ingles"].unique())

array(['56', '48', '42', '43', '62', '37', '50', '64', '66', '53', '57',
       '44', '54', '63', '46', '60', '49', '39', '52', '32', '68', '47',
       '51', '61', '38', '55', '82', '41', '34', '58', '74', '65', '59',
       '36', '73', '31', '78', '35', '67', '81', '76', '71', '33', '45',
       '77', '69', '40', '29', '70', '30', '72', '100', '75', '79', '80',
       '83', '85', '84', '0', '28', nan], dtype=object)

Antes de imputar el promedio  en esta columna vamos a convertir correctamente los tipos de cada una:

In [29]:
#Definir el tipo de cada variable:
dffinal['periodo'] = dffinal['periodo'].astype('category')

columnas_category = [
    'cole_area_ubicacion',
    'cole_bilingue',
    'cole_calendario',
    'cole_caracter',
    'cole_genero',
    'cole_jornada',
    'cole_naturaleza',    
    'estu_genero',
    'fami_cuartoshogar',
    'fami_educacionmadre',
    'fami_educacionpadre',
    'fami_estratovivienda',
    'fami_personashogar',
    'fami_tieneautomovil',
    'fami_tienecomputador',
    'fami_tieneinternet',
    'fami_tienelavadora',
    'desemp_ingles'
]
for col in columnas_category:
    dffinal[col] = dffinal[col].astype('category')

columnas_int = [
    'punt_ingles',
    'punt_matematicas',
    'punt_sociales_ciudadanas',
    'punt_c_naturales',
    'punt_lectura_critica',
    'punt_global'
]
for col in columnas_int:
    dffinal[col] = pd.to_numeric(dffinal[col], errors='coerce').astype('Int64')

In [30]:
dffinal.dtypes

periodo                     category
cole_area_ubicacion         category
cole_bilingue               category
cole_calendario             category
cole_caracter               category
cole_genero                 category
cole_jornada                category
cole_naturaleza             category
estu_genero                 category
fami_cuartoshogar           category
fami_educacionmadre         category
fami_educacionpadre         category
fami_estratovivienda        category
fami_personashogar          category
fami_tieneautomovil         category
fami_tienecomputador        category
fami_tieneinternet          category
fami_tienelavadora          category
desemp_ingles               category
punt_ingles                    Int64
punt_matematicas               Int64
punt_sociales_ciudadanas       Int64
punt_c_naturales               Int64
punt_lectura_critica           Int64
punt_global                    Int64
dtype: object

In [31]:
resumen_nulos = pd.DataFrame({
    "nulos": dffinal.isna().sum(),
    "porcentaje_nulos": (dffinal.isna().mean() * 100).round(2)
})

resumen_nulos = resumen_nulos.query("nulos > 0")
resumen_nulos = resumen_nulos.sort_values(
    "porcentaje_nulos",
    ascending=False
)

display(resumen_nulos)

,nulos,porcentaje_nulos
punt_ingles,15,0.02


In [32]:
promedio_ingles = int(round(dffinal["punt_ingles"].mean()))
dffinal["punt_ingles"] = dffinal["punt_ingles"].fillna(promedio_ingles)
print("Promedio utilizado:", promedio_ingles)
print("Tipo de dato:", dffinal["punt_ingles"].dtype)
print("Nulos restantes:", dffinal["punt_ingles"].isna().sum())

Promedio utilizado: 56
Tipo de dato: Int64
Nulos restantes: 0


Finalmente, se cuenta con una base depurada de 76.253 registros y 25 variables:

In [33]:
dffinal.info()

<class 'pandas.core.frame.DataFrame'>
Index: 76253 entries, 0 to 152582
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   periodo                   76253 non-null  category
 1   cole_area_ubicacion       76253 non-null  category
 2   cole_bilingue             76253 non-null  category
 3   cole_calendario           76253 non-null  category
 4   cole_caracter             76253 non-null  category
 5   cole_genero               76253 non-null  category
 6   cole_jornada              76253 non-null  category
 7   cole_naturaleza           76253 non-null  category
 8   estu_genero               76253 non-null  category
 9   fami_cuartoshogar         76253 non-null  category
 10  fami_educacionmadre       76253 non-null  category
 11  fami_educacionpadre       76253 non-null  category
 12  fami_estratovivienda      76253 non-null  category
 13  fami_personashogar        76253 non-null  category

# Análisis descriptivo y exploración gráfica - Kevin

# Relaciones entre variables y hallazgos exploratorios - Andrés